# WP41 — Strange Loop Visualiser (v0.4)
## HierarchyLevel · CrossLevelSignal · StrangeLoopTrace · LoopReport

Makes the tangled hierarchy of the CRLS stack *observable* as a live artefact.

Every signal crossing between the three levels of the CRLS strange loop is
logged, the entanglement matrix is computed, bidirectional loops are detected,
and the **self-observation moment** is identified — the generation at which the
system is literally observing its own observation.

> *"In a strange loop, moving through the levels of a hierarchical system,  
> we unexpectedly find ourselves back where we started."*  
> — Douglas Hofstadter, *Gödel, Escher, Bach* (1979)

Runtime: **< 1 min** (pure Python, no GPU)

In [ ]:
import sys, os, importlib
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('Prometheus_v0_PoC'):
        os.system('git clone -b wp16-notebook-only https://github.com/pmcray/Prometheus_v0_PoC.git')
    os.chdir('Prometheus_v0_PoC'); sys.path.insert(0, '/content/Prometheus_v0_PoC')
    importlib.invalidate_caches()
else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if repo_root not in sys.path: sys.path.insert(0, repo_root)
    importlib.invalidate_caches()
import warnings; warnings.filterwarnings('ignore')
import math, random, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.patches import FancyArrowPatch
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.figsize': (16, 7), 'font.size': 11})
SEED = 42; random.seed(SEED); np.random.seed(SEED)
import prometheus; print(f'Prometheus {prometheus.__version__} OK')

In [ ]:
from prometheus.wp41_strange_loop_visualiser import (
    HierarchyLevel, CrossLevelSignal, StrangeLoopTrace,
    StrangeLoopSimulator, LoopReport, run_loop,
    verify_wp41_exit_criteria,
)
print('WP41 imports OK')
print()
print('Hierarchy levels:')
for lvl in HierarchyLevel:
    print(f'  {lvl.value}')

In [ ]:
# ── Run the instrumented simulator for 120 generations
N_GEN  = 120
report = run_loop(n_generations=N_GEN, seed=42)
print(report.summary())

In [ ]:
# ── Signal breakdown by WP module and direction
from collections import Counter
sig_by_wp  = Counter(s.wp_module   for s in report.trace.signals())
sig_by_dir = Counter(
    'upward'     if s.is_upward()   else
    'downward'   if s.is_downward() else
    'same-level'
    for s in report.trace.signals()
)
print('Signals by WP module:')
for wp, count in sorted(sig_by_wp.items()):
    print(f'  {wp:6s}  {count:5d}')
print()
print('Signals by direction:')
for direction, count in sig_by_dir.items():
    print(f'  {direction:<12}  {count:5d}')
print()
print('Detected loops (bidirectional level pairs):')
for (a, b) in report.detected_loops:
    print(f'  {a.value}  ↔  {b.value}')

In [ ]:
# ── The Hofstadter moment
g_star = report.self_observation_gen
print(f'Self-observation moment: generation {g_star}')
print()
if g_star is not None:
    # Show the signals at g_star - 1 and g_star
    L0 = HierarchyLevel.OBJECT_LEVEL
    L1 = HierarchyLevel.META_LEVEL
    L2 = HierarchyLevel.META_META_LEVEL
    
    prev_sigs = report.trace.signals(source=L1, target=L0)
    prev_sigs = [s for s in prev_sigs if s.generation == g_star - 1]
    curr_sigs = report.trace.signals(source=L0, target=L1)
    curr_sigs = [s for s in curr_sigs if s.generation == g_star]
    
    print(f'Generation {g_star-1}: META_LEVEL → OBJECT_LEVEL')
    for s in prev_sigs:
        print(f'  [{s.wp_module}] {s.signal_name}  value={s.value:.4f}')
    print()
    print(f'Generation {g_star}: OBJECT_LEVEL → META_LEVEL  ← the echo')
    for s in curr_sigs:
        print(f'  [{s.wp_module}] {s.signal_name}  value={s.value:.4f}')
    print()
    print('At this moment, the META_LEVEL is observing a state that was')
    print('itself shaped by the META_LEVEL\'s previous synthesis action.')
    print()
    print('"This system is currently observing its own observation."')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 7))

L0 = HierarchyLevel.OBJECT_LEVEL
L1 = HierarchyLevel.META_LEVEL
L2 = HierarchyLevel.META_META_LEVEL
levels_ordered = [L0, L1, L2]
level_labels   = ['L0\nObject', 'L1\nMeta', 'L2\nMeta-Meta']
level_colors   = ['#1565C0', '#2E7D32', '#6A1B9A']

# ── Panel A: entanglement heatmap
ax = axes[0]
E  = report.entanglement_matrix
import numpy as np
E_arr = np.array(E)
im = ax.imshow(E_arr, cmap='YlOrRd', vmin=0, aspect='auto')
ax.set_xticks([0,1,2]); ax.set_yticks([0,1,2])
ax.set_xticklabels(level_labels); ax.set_yticklabels(level_labels)
ax.set_xlabel('Target level'); ax.set_ylabel('Source level')
ax.set_title('Entanglement Matrix E[source][target]\n'
             '(off-diagonal = cross-level coupling)', fontweight='bold')
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{E[i][j]:.3f}', ha='center', va='center',
                fontsize=12, color='black' if E[i][j] < 0.3 else 'white',
                fontweight='bold')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Normalised MI proxy')

# ── Panel B: signal flow diagram (animated per generation — static version)
ax2 = axes[1]
ax2.set_xlim(-0.5, 2.5); ax2.set_ylim(-0.5, 3.5); ax2.axis('off')
ax2.set_title('Cross-Level Signal Flow\n(thickness = signal volume)', fontweight='bold')

# Draw level boxes
for i, (lbl, col) in enumerate(zip(level_labels, level_colors)):
    rect = plt.Rectangle((i - 0.35, 1.3), 0.7, 0.9, color=col, alpha=0.25, zorder=1)
    ax2.add_patch(rect)
    ax2.text(i, 1.75, lbl, ha='center', va='center', fontsize=11,
             color=col, fontweight='bold')

# Count signals per (src→tgt) pair
from collections import defaultdict
pair_counts = defaultdict(int)
pair_values = defaultdict(float)
for s in report.trace.signals():
    key = (levels_ordered.index(s.source), levels_ordered.index(s.target))
    pair_counts[key] += 1
    pair_values[key] += abs(s.value)
max_count = max(pair_counts.values()) if pair_counts else 1

arrow_props = dict(arrowstyle='->', connectionstyle='arc3,rad=0.3',
                   mutation_scale=15, zorder=3)
for (si, ti), count in pair_counts.items():
    if si == ti:
        continue
    sx, sy = si, 1.75
    tx, ty = ti, 1.75
    lw = 0.5 + 4.0 * count / max_count
    col = level_colors[si]
    ax2.annotate('', xy=(tx, ty), xytext=(sx, sy),
                 arrowprops=dict(**arrow_props, lw=lw, color=col))
    mx = (sx + tx) / 2 + (0.15 if ty > sy else -0.15)
    my = (sy + ty) / 2 + (0.45 if si < ti else -0.45)
    ax2.text(mx, my, str(count), ha='center', va='center', fontsize=8, color=col)

# Annotate WP modules
wp_annotations = [
    (0.5,  2.7, 'WP19 (ATE)',    '#1E88E5'),
    (0.5,  2.4, 'WP22 (UCB1)',   '#1E88E5'),
    (1.5,  2.7, 'WP21 (meta-∇)', '#7B1FA2'),
    (1.5,  2.4, 'WP40 (safety)', '#7B1FA2'),
    (0.5,  0.9, 'WP17 (synth)',  '#E53935'),
]
for x, y, label, c in wp_annotations:
    ax2.text(x, y, label, ha='center', va='center', fontsize=8.5, color=c,
             bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor=c, alpha=0.8))

# Self-observation marker
if g_star is not None:
    ax2.text(1.0, 0.3,
             f'⟳ Self-observation\n  at generation {g_star}',
             ha='center', va='center', fontsize=10, color='#B71C1C',
             fontweight='bold',
             bbox=dict(boxstyle='round', facecolor='#FFEBEE', edgecolor='#E53935'))

# ── Panel C: signal volume timeline
ax3 = axes[2]
# Bin signals into windows of 10 generations
window = 10
n_windows = N_GEN // window
up_series   = []
down_series = []
gen_mids    = []
for w in range(n_windows):
    g_start = w * window
    g_end   = g_start + window
    sigs_w  = [s for s in report.trace.signals() if g_start <= s.generation < g_end]
    up_series.append(sum(1 for s in sigs_w if s.is_upward()))
    down_series.append(sum(1 for s in sigs_w if s.is_downward()))
    gen_mids.append(g_start + window // 2)

ax3.fill_between(gen_mids, up_series,   alpha=0.5, color='#1E88E5', label='Upward (L→L+1)')
ax3.fill_between(gen_mids, down_series, alpha=0.5, color='#E53935', label='Downward (L+1→L)')
ax3.plot(gen_mids, up_series,   'b-', lw=2)
ax3.plot(gen_mids, down_series, 'r-', lw=2)

if g_star is not None:
    ax3.axvline(g_star, color='#B71C1C', linestyle='--', lw=2,
                label=f'Self-observation (gen {g_star})')

ax3.set_xlabel('Generation')
ax3.set_ylabel('Signal count per 10-gen window')
ax3.set_title('Upward vs Downward Signal Volume\n(balance = tangling measure)', fontweight='bold')
ax3.legend(fontsize=9)

fig.suptitle('WP41: Strange Loop Visualiser — The Tangled Hierarchy Made Observable',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('wp41_strange_loop_visualiser.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved wp41_strange_loop_visualiser.png')

In [ ]:
criteria = verify_wp41_exit_criteria(report)
print('WP41 Exit Criteria Verification'); print('=' * 62)
for c, ok in criteria.items():
    print(f'  {"PASS" if ok else "FAIL"}  {c}')
if all(criteria.values()):
    print()
    print('All WP41 exit criteria satisfied.')
    print()
    print('Hofstadter statement:')
    print(f'  {report.hofstadter_statement}')

---
## Conclusions

**WP41** makes the CRLS strange loop legible as a first-class observable object:

### The three levels

| Level | WP modules | Role |
|-------|-----------|------|
| **L0 Object** | WP17 | Strategy probs, raw accuracy |
| **L1 Meta** | WP19, WP22 | Synthesis decisions, causal attribution |
| **L2 Meta-Meta** | WP21, WP40 | Hyperparameter updates, safety gating |

### What makes it a *strange* loop (not merely a *nested* one)

A nested hierarchy has signals flowing in one direction only.  The entanglement
matrix shows **significant off-diagonal mass in both triangles**: L2 feeds back
to L1 (safety decisions), L1 feeds back to L0 (synthesis actions), *and* L0's
outcomes feed back to L2 (meta-gradient updates).  The hierarchy is genuinely
tangled.

### The self-observation moment

At the detected generation, the META_LEVEL observes an object-level state that
was *itself produced in response to the META_LEVEL's previous synthesis action*.
This is Hofstadter's definition of a strange loop, realised in running Python.

> *"This system is currently observing its own observation."*

### References
- Hofstadter, *Gödel, Escher, Bach* (1979) — Strange loops, tangled hierarchies
- Hofstadter, *I Am a Strange Loop* (2007) — Self-models within systems
- Good (1965) — Loop closure as prerequisite for intelligence explosion